#Importing Libraries

In [4]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler, MinMaxScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import log, col

#Starting Spark Session

In [7]:
# Initialize Spark session
spark = SparkSession.builder.master("local[1]") \
    .appName("Diabetes") \
    .getOrCreate()

spark

#MetaData

In [10]:
# Read the CSV file into a DataFrame
data = spark.read.csv("C:/Users/showg/OneDrive/Desktop/Big Data/Diabetes.csv", header=True, inferSchema=True)

# Display schema and data
data.printSchema()

# Total number of rows
total_rows = data.count()
print(f"Total number of rows: {total_rows}")

# Total number of columns
total_columns = len(data.columns)
print(f"Total number of columns: {total_columns}")

# Unique counts for each column
print("Unique value counts for each column:")
for column in data.columns:
    unique_count = data.select(column).distinct().count()
    print(f"Column '{column}' has {unique_count} unique values.")

root
 |-- TOTAL_QUANTITY: integer (nullable = true)
 |-- PRACTICE_CODE: string (nullable = true)
 |-- ITEMS: integer (nullable = true)
 |-- CHEMICAL_SUBSTANCE_BNF_DESCR: string (nullable = true)
 |-- BNF_CHAPTER_PLUS_CODE: string (nullable = true)
 |-- POSTCODE: string (nullable = true)
 |-- YEAR_MONTH: string (nullable = true)
 |-- NIC: double (nullable = true)
 |-- ADQUSAGE: double (nullable = true)
 |-- PRACTICE_NAME: string (nullable = true)
 |-- ACTUAL_COST: double (nullable = true)
 |-- BNF_DESCRIPTION: string (nullable = true)
 |-- QUANTITY: integer (nullable = true)
 |-- PRACTICE_LATITUDE: double (nullable = true)
 |-- PRACTICE_LONGITUDE: double (nullable = true)

Total number of rows: 14078
Total number of columns: 15
Unique value counts for each column:
Column 'TOTAL_QUANTITY' has 736 unique values.
Column 'PRACTICE_CODE' has 31 unique values.
Column 'ITEMS' has 357 unique values.
Column 'CHEMICAL_SUBSTANCE_BNF_DESCR' has 4 unique values.
Column 'BNF_CHAPTER_PLUS_CODE' has 1 

#Data Description

In [17]:
from pyspark.sql.functions import col, countDistinct, min, max
from tabulate import tabulate


variable_summary = []

# List of all columns in the dataset
columns = data.columns


for column in columns:
    data_type = dict(data.dtypes).get(column)
    summary = {"Variable": column, "Data Type": data_type}
    
    if data_type in ["int", "double"]:  # Numerical columns
        # Calculate range (min and max)
        min_val = data.select(min(col(column))).collect()[0][0]
        max_val = data.select(max(col(column))).collect()[0][0]
        summary["Range"] = f"({min_val}, {max_val})"
        summary["Cardinality"] = "N/A"  # Not applicable for numerical columns
    
    elif data_type == "string":  # Categorical columns
        # Calculate cardinality (number of unique values)
        cardinality = data.select(countDistinct(col(column))).collect()[0][0]
        summary["Range"] = "N/A"  # Not applicable for categorical columns
        summary["Cardinality"] = cardinality
    
    variable_summary.append(summary)

# Convert dictionaries to a list of rows for tabulate
table_data = [[item["Variable"], item["Data Type"], item["Range"], item["Cardinality"]] for item in variable_summary]
headers = ["Variable", "Data Type", "Range", "Cardinality"]

# Print the table
print(tabulate(table_data, headers=headers, tablefmt="grid"))


+------------------------------+-------------+----------------------------+---------------+
| Variable                     | Data Type   | Range                      | Cardinality   |
+==============================+=============+============================+===============+
| TOTAL_QUANTITY               | int         | (1, 58688)                 | N/A           |
+------------------------------+-------------+----------------------------+---------------+
| PRACTICE_CODE                | string      | N/A                        | 31            |
+------------------------------+-------------+----------------------------+---------------+
| ITEMS                        | int         | (1, 741)                   | N/A           |
+------------------------------+-------------+----------------------------+---------------+
| CHEMICAL_SUBSTANCE_BNF_DESCR | string      | N/A                        | 4             |
+------------------------------+-------------+----------------------------+-----

#Imputing Null Values

In [18]:

file_path = "C:/Users/showg/OneDrive/Desktop/Big Data/Diabetes.csv"
df = spark.read.csv(file_path, header=True, inferSchema=True)

# default values for the columns where PRACTICE_CODE = 'F84714'
default_values = df.filter(col("PRACTICE_CODE") == "F84714") \
                   .select("POSTCODE", "PRACTICE_LATITUDE", "PRACTICE_LONGITUDE") \
                   .first()

# Extract default values
if default_values:
    default_postcode = default_values["POSTCODE"]
    default_latitude = default_values["PRACTICE_LATITUDE"]
    default_longitude = default_values["PRACTICE_LONGITUDE"]
else:
    raise ValueError("No rows found with PRACTICE_CODE = 'F84714'")

# Replace null values in the respective columns with the default values
df = df.fillna({
    "POSTCODE": default_postcode,
    "PRACTICE_LATITUDE": default_latitude,
    "PRACTICE_LONGITUDE": default_longitude
})

# Filter rows where changes were made (nulls replaced with default values)
df_changes = df

# Display the changed rows
df_changes.show()


+--------------+-------------+-----+----------------------------+---------------------+--------+----------+------+----------+--------------------+-----------+--------------------+--------+-----------------+------------------+
|TOTAL_QUANTITY|PRACTICE_CODE|ITEMS|CHEMICAL_SUBSTANCE_BNF_DESCR|BNF_CHAPTER_PLUS_CODE|POSTCODE|YEAR_MONTH|   NIC|  ADQUSAGE|       PRACTICE_NAME|ACTUAL_COST|     BNF_DESCRIPTION|QUANTITY|PRACTICE_LATITUDE|PRACTICE_LONGITUDE|
+--------------+-------------+-----+----------------------------+---------------------+--------+----------+------+----------+--------------------+-----------+--------------------+--------+-----------------+------------------+
|         15680|       F84031|   70|        Metformin hydroch...| 06: Endocrine System|  E1 0LS|    23-Jul| 442.4|5226.66666|THE JUBILEE STREE...|  415.30817|Metformin 500mg t...|     224|       51.5135999|          -0.05055|
|           112|       F84087|    1|        Metformin hydroch...| 06: Endocrine System|  E1 4FG|

#Deleting Unwanted Columns

In [21]:
# Drop the specified columns
columns_to_drop = ["POSTCODE", "BNF_CHAPTER_PLUS_CODE", "PRACTICE_CODE", "PRACTICE_NAME"]
df_updated = df_changes.drop(*columns_to_drop)

df_updated.show()


+--------------+-----+----------------------------+----------+------+----------+-----------+--------------------+--------+-----------------+------------------+
|TOTAL_QUANTITY|ITEMS|CHEMICAL_SUBSTANCE_BNF_DESCR|YEAR_MONTH|   NIC|  ADQUSAGE|ACTUAL_COST|     BNF_DESCRIPTION|QUANTITY|PRACTICE_LATITUDE|PRACTICE_LONGITUDE|
+--------------+-----+----------------------------+----------+------+----------+-----------+--------------------+--------+-----------------+------------------+
|         15680|   70|        Metformin hydroch...|    23-Jul| 442.4|5226.66666|  415.30817|Metformin 500mg t...|     224|       51.5135999|          -0.05055|
|           112|    1|        Metformin hydroch...|    23-Jul|  3.84|  37.33333|    3.70971|Glucophage 500mg ...|     112|      51.51835505|       -0.03830635|
|          5152|   23|        Metformin hydroch...|    23-Jul|145.36|1717.33333|   136.4584|Metformin 500mg t...|     224|       51.5363018|      -0.027021075|
|          1064|   76|        Metformin 

#Saving Cleaned |Dataset

In [24]:
import pandas as pd

# Convert PySpark DataFrame to pandas DataFrame
df_updated_pandas = df_updated.toPandas()

# Perform pandas operations
print(df_updated_pandas.head())

# Save the pandas DataFrame as a CSV file
df_updated_pandas.to_csv("C:/Users/showg/OneDrive/Desktop/Big Data/df_updated.csv", index=False)

   TOTAL_QUANTITY  ITEMS CHEMICAL_SUBSTANCE_BNF_DESCR YEAR_MONTH     NIC  \
0           15680     70      Metformin hydrochloride     23-Jul  442.40   
1             112      1      Metformin hydrochloride     23-Jul    3.84   
2            5152     23      Metformin hydrochloride     23-Jul  145.36   
3            1064     76      Metformin hydrochloride     23-Jul   30.40   
4              56      1      Metformin hydrochloride     23-Jul    1.66   

     ADQUSAGE  ACTUAL_COST                           BNF_DESCRIPTION  \
0  5226.66666    415.30817                   Metformin 500mg tablets   
1    37.33333      3.70971                  Glucophage 500mg tablets   
2  1717.33333    136.45840                   Metformin 500mg tablets   
3   354.66667     37.02111                   Metformin 500mg tablets   
4    18.66667      1.56749  Metformin 500mg modified-release tablets   

   QUANTITY  PRACTICE_LATITUDE  PRACTICE_LONGITUDE  
0       224          51.513600           -0.050550  
1   

#Seperating "YEAR" and "MONTH" Column

In [27]:
from pyspark.sql.functions import col, when, lit, upper

# Convert YEAR_MONTH to uppercase to standardize abbreviations
df_updated = df_updated.withColumn("YEAR_MONTH", upper(col("YEAR_MONTH")))

# Extract YEAR as a numeric value
df_updated = df_updated.withColumn("YEAR", (col("YEAR_MONTH").substr(1, 2).cast("int") + 2000))

# Initialize MONTH column with null values
df_updated = df_updated.withColumn("MONTH", lit(None).cast("int"))

# Map MONTH abbreviations to numeric values
month_mapping = {
    "JAN": 1, "FEB": 2, "MAR": 3, "APR": 4, "MAY": 5, "JUN": 6,
    "JUL": 7, "AUG": 8, "SEP": 9, "OCT": 10, "NOV": 11, "DEC": 12
}

# Update MONTH column iteratively
for month, num in month_mapping.items():
    df_updated = df_updated.withColumn(
        "MONTH", when(col("YEAR_MONTH").like(f"%{month}"), lit(num)).otherwise(col("MONTH"))
    )

# Drop the original YEAR_MONTH column 
df_updated = df_updated.drop("YEAR_MONTH")

# Display the updated DataFrame
df_updated.show(truncate=False)



+--------------+-----+----------------------------+------+----------+-----------+----------------------------------------+--------+-----------------+------------------+----+-----+
|TOTAL_QUANTITY|ITEMS|CHEMICAL_SUBSTANCE_BNF_DESCR|NIC   |ADQUSAGE  |ACTUAL_COST|BNF_DESCRIPTION                         |QUANTITY|PRACTICE_LATITUDE|PRACTICE_LONGITUDE|YEAR|MONTH|
+--------------+-----+----------------------------+------+----------+-----------+----------------------------------------+--------+-----------------+------------------+----+-----+
|15680         |70   |Metformin hydrochloride     |442.4 |5226.66666|415.30817  |Metformin 500mg tablets                 |224     |51.5135999       |-0.05055          |2023|7    |
|112           |1    |Metformin hydrochloride     |3.84  |37.33333  |3.70971    |Glucophage 500mg tablets                |112     |51.51835505      |-0.03830635       |2023|7    |
|5152          |23   |Metformin hydrochloride     |145.36|1717.33333|136.4584   |Metformin 500mg tab

#Summary Statistics

In [32]:
from pyspark.sql.functions import col, mean, stddev, sum, skewness, kurtosis

# List of numerical columns for summary statistics
numerical_columns = ["TOTAL_QUANTITY", "ITEMS", "NIC", "ADQUSAGE", "ACTUAL_COST", "QUANTITY", 
                     "PRACTICE_LATITUDE", "PRACTICE_LONGITUDE", "YEAR", "MONTH"]

# Initialize 
statistics = []

# statistics for each column
for column in numerical_columns:
    stats = (
        df_updated
        .select(
            sum(col(column)).alias("sum"),
            mean(col(column)).alias("mean"),
            stddev(col(column)).alias("stddev"),
            skewness(col(column)).alias("skewness"),
            kurtosis(col(column)).alias("kurtosis"),
        )
        .withColumn("variable", lit(column))
    )
    statistics.append(stats)

# statistics into a single DataFrame
summary_stats = statistics[0]
for stat in statistics[1:]:
    summary_stats = summary_stats.union(stat)

# Show summary statistics
summary_stats.select("variable", "sum", "mean", "stddev", "skewness", "kurtosis").show(truncate=False)


+------------------+------------------+--------------------+--------------------+--------------------+--------------------+
|variable          |sum               |mean                |stddev              |skewness            |kurtosis            |
+------------------+------------------+--------------------+--------------------+--------------------+--------------------+
|TOTAL_QUANTITY    |1.9165131E7       |1361.3532461997443  |3531.362804896607   |6.172432820053785   |54.2078582910977    |
|ITEMS             |292637.0          |20.786830515698252  |52.08330663770651   |6.154440065112511   |50.87498331093555   |
|NIC               |643339.8499999918 |45.698241937774675  |97.80824196263892   |5.315611942413339   |43.33897129727307   |
|ADQUSAGE          |6686488.362940015 |474.96010533740696  |1176.3442149298069  |6.1360631094465115  |53.91769890820288   |
|ACTUAL_COST       |601922.4330299995 |42.75624613084241   |88.33369087584377   |5.188238850691052   |41.06265411484201   |
|QUANTIT

#Correlation Ananlysis

In [37]:
from pyspark.sql.functions import col

# Numerical columns to calculate correlation with the target variable
numerical_columns = ["TOTAL_QUANTITY", "ITEMS", "NIC", "ADQUSAGE", "ACTUAL_COST", "QUANTITY", 
                     "PRACTICE_LATITUDE", "PRACTICE_LONGITUDE", "YEAR", "MONTH"]
target_variable = "ACTUAL_COST"

# Initialize an empty list to store correlations
correlations = []

# correlation for each column with the target variable
for column in numerical_columns:
    try:
        correlation_value = df_updated.stat.corr(column, target_variable)
        if correlation_value is not None:  # Ensure valid correlation value
            correlations.append((column, correlation_value))
        else:
            print(f"Correlation for {column} is None. Skipping.")
    except Exception as e:
        print(f"Error calculating correlation for {column}: {e}")

# Print the correlations list
print("Correlations List:", correlations)

Correlations List: [('TOTAL_QUANTITY', 0.9030097156355491), ('ITEMS', 0.5323701107770327), ('NIC', 0.9966202259645706), ('ADQUSAGE', 0.8976584338732689), ('ACTUAL_COST', 1.0), ('QUANTITY', 0.23411480188813713), ('PRACTICE_LATITUDE', -0.004143185257360765), ('PRACTICE_LONGITUDE', -0.0356790067120478), ('YEAR', -0.013086830156614885), ('MONTH', 0.012760111139862433)]


#PIPELINE 1

In [40]:
log_target_column = "LOG_ACTUAL_COST"
df_updated = df_updated.withColumn(log_target_column, log(col("ACTUAL_COST") + 1))

#StringIndexer and OneHotEncoder for categorical variables
indexer = StringIndexer(inputCol="BNF_DESCRIPTION", outputCol="BNF_DESCRIPTION_INDEX")
encoder = OneHotEncoder(inputCol="BNF_DESCRIPTION_INDEX", outputCol="BNF_DESCRIPTION_OHE")
chem_indexer = StringIndexer(inputCol="CHEMICAL_SUBSTANCE_BNF_DESCR", outputCol="CHEMICAL_SUBSTANCE_BNF_DESCR_INDEX")

#Log-transform numerical features
log_columns = ["TOTAL_QUANTITY", "NIC", "ADQUSAGE"]
for col_name in log_columns:
    df_updated = df_updated.withColumn(f"LOG_{col_name}", log(col(col_name) + 1))

#Assemble and scale log-transformed features
scaled_features = []
for col_name in log_columns:
    assembler = VectorAssembler(inputCols=[f"LOG_{col_name}"], outputCol=f"{col_name}_vec")
    scaler = StandardScaler(inputCol=f"{col_name}_vec", outputCol=f"{col_name}_scaled", withMean=True, withStd=True)
    scaled_features.append(assembler)
    scaled_features.append(scaler)

#MinMax scaling for QUANTITY, YEAR, MONTH
minmax_columns = ["QUANTITY", "YEAR", "MONTH"]
minmax_scalers = []
for column in minmax_columns:
    assembler = VectorAssembler(inputCols=[column], outputCol=f"{column}_vec")
    scaler = MinMaxScaler(inputCol=f"{column}_vec", outputCol=f"{column}_scaled")
    minmax_scalers.append(assembler)
    minmax_scalers.append(scaler)

#KMeans clustering for geolocation
geo_assembler = VectorAssembler(inputCols=["PRACTICE_LATITUDE", "PRACTICE_LONGITUDE"], outputCol="geo_features")

#VectorAssembler for all features
final_features = [
    "BNF_DESCRIPTION_OHE", "CHEMICAL_SUBSTANCE_BNF_DESCR_INDEX", "TOTAL_QUANTITY_scaled", 
    "NIC_scaled", "ADQUSAGE_scaled", "QUANTITY_scaled", "YEAR_scaled", "MONTH_scaled"
]
feature_assembler = VectorAssembler(inputCols=final_features, outputCol="features")

#Linear Regression
lr = LinearRegression(featuresCol="features", labelCol=log_target_column)

# Pipeline assembly
stages = [
    indexer, encoder, chem_indexer,
    *scaled_features, *minmax_scalers,
    geo_assembler, feature_assembler, lr
]
log_pipeline = Pipeline(stages=stages)

# Train-test split
train_data, test_data = df_updated.randomSplit([0.8, 0.2], seed=42)

# Fit the pipeline and evaluate the model
log_model = log_pipeline.fit(train_data)
log_predictions = log_model.transform(test_data)

#Evaluate model performance
evaluator_rmse = RegressionEvaluator(labelCol=log_target_column, predictionCol="prediction", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol=log_target_column, predictionCol="prediction", metricName="r2")

rmse = evaluator_rmse.evaluate(log_predictions)
r2 = evaluator_r2.evaluate(log_predictions)

print(f"Root Mean Squared Error (RMSE): {rmse}")
print(f"R-squared (R2): {r2}")

# Display sample predictions
log_predictions.select("prediction", log_target_column, "features").show(truncate=False)


Root Mean Squared Error (RMSE): 0.09353342859553959
R-squared (R2): 0.9948099942291405
+-------------------+-------------------+------------------------------------------------------------------------------------------------------------------------------------------+
|prediction         |LOG_ACTUAL_COST    |features                                                                                                                                  |
+-------------------+-------------------+------------------------------------------------------------------------------------------------------------------------------------------+
|0.2611429621260317 |0.1315456723864203 |(34,[0,28,29,30,33],[1.0,-3.560615752776987,-2.0683937934560626,-3.188151135052496,0.9090909090909092])                                   |
|0.29306607384886574|0.1877236060829493 |(34,[0,28,29,30,31,32,33],[1.0,-2.9367093519516847,-2.0192556853218617,-2.805930129851636,0.0025020850708924102,1.0,0.18181818181818182]) |
|0.30902

In [42]:
# Convert the final transformed DataFrame to pandas DataFrame
final_df = log_predictions.toPandas()

# Save the pandas DataFrame to a CSV file
file_name = "transformed_data.csv"
final_df.to_csv(file_name, index=False)

print(f"Transformed data saved to {file_name}")


Transformed data saved to transformed_data.csv


#PIPELINE 2

In [24]:
# Linear Regression for ACTUAL_COST
lr_original = LinearRegression(featuresCol="features", labelCol="ACTUAL_COST", predictionCol="prediction")

# Define the pipeline for original target variable
original_pipeline = Pipeline(stages=stages[:-1] + [lr_original])  # Replace only the LinearRegression stage

# Fit and evaluate the model
original_model = original_pipeline.fit(train_data)
original_predictions = original_model.transform(test_data)

evaluator_rmse = RegressionEvaluator(labelCol="ACTUAL_COST", predictionCol="prediction", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol="ACTUAL_COST", predictionCol="prediction", metricName="r2")

rmse_original = evaluator_rmse.evaluate(original_predictions)
r2_original = evaluator_r2.evaluate(original_predictions)

print(f"Original Target Variable Metrics:")
print(f"Root Mean Squared Error (RMSE): {rmse_original}")
print(f"R²: {r2_original}")


Original Target Variable Metrics:
Root Mean Squared Error (RMSE): 56.24526007473526
R²: 0.6300262836881126


#The code from here on, are all included in the above Pipeline1. This codes were created especially to show the results individually for the purpose of this report.

#The code from here on, are all included in the above Pipeline1. This codes were created especially to show the results individually for the purpose of this report.

In [16]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder

#Index the BNF_DESCRIPTION column (required before OneHotEncoding)
indexer = StringIndexer(inputCol="BNF_DESCRIPTION", outputCol="BNF_DESCRIPTION_INDEX")
df_indexed = indexer.fit(df_updated).transform(df_updated)

#Apply OneHotEncoder to the indexed column
encoder = OneHotEncoder(inputCol="BNF_DESCRIPTION_INDEX", outputCol="BNF_DESCRIPTION_OHE")
df_encoded = encoder.fit(df_indexed).transform(df_indexed)

# Display the updated DataFrame with the new column
df_encoded.select("BNF_DESCRIPTION", "BNF_DESCRIPTION_INDEX", "BNF_DESCRIPTION_OHE").show(truncate=False)


+----------------------------------------+---------------------+-------------------+
|BNF_DESCRIPTION                         |BNF_DESCRIPTION_INDEX|BNF_DESCRIPTION_OHE|
+----------------------------------------+---------------------+-------------------+
|Metformin 500mg tablets                 |0.0                  |(27,[0],[1.0])     |
|Glucophage 500mg tablets                |19.0                 |(27,[19],[1.0])    |
|Metformin 500mg tablets                 |0.0                  |(27,[0],[1.0])     |
|Metformin 500mg tablets                 |0.0                  |(27,[0],[1.0])     |
|Metformin 500mg modified-release tablets|1.0                  |(27,[1],[1.0])     |
|Metformin 500mg tablets                 |0.0                  |(27,[0],[1.0])     |
|Metformin 500mg tablets                 |0.0                  |(27,[0],[1.0])     |
|Metformin 500mg tablets                 |0.0                  |(27,[0],[1.0])     |
|Metformin 500mg modified-release tablets|1.0                  |(

In [18]:
from pyspark.ml.feature import StringIndexer

# StringIndexer to CHEMICAL_SUBSTANCE_BNF_DESCR
chem_indexer = StringIndexer(inputCol="CHEMICAL_SUBSTANCE_BNF_DESCR", outputCol="CHEMICAL_SUBSTANCE_BNF_DESCR_INDEX")
df_encoded = chem_indexer.fit(df_encoded).transform(df_encoded)

# Display the original and indexed columns
df_encoded.select("CHEMICAL_SUBSTANCE_BNF_DESCR", "CHEMICAL_SUBSTANCE_BNF_DESCR_INDEX").show(truncate=False)


+----------------------------+----------------------------------+
|CHEMICAL_SUBSTANCE_BNF_DESCR|CHEMICAL_SUBSTANCE_BNF_DESCR_INDEX|
+----------------------------+----------------------------------+
|Metformin hydrochloride     |0.0                               |
|Metformin hydrochloride     |0.0                               |
|Metformin hydrochloride     |0.0                               |
|Metformin hydrochloride     |0.0                               |
|Metformin hydrochloride     |0.0                               |
|Metformin hydrochloride     |0.0                               |
|Metformin hydrochloride     |0.0                               |
|Metformin hydrochloride     |0.0                               |
|Metformin hydrochloride     |0.0                               |
|Metformin hydrochloride     |0.0                               |
|Metformin hydrochloride     |0.0                               |
|Metformin hydrochloride     |0.0                               |
|Metformin

In [20]:
from pyspark.sql.functions import rand

# Shuffle the rows randomly and display a subset
df_encoded.orderBy(rand()).select("CHEMICAL_SUBSTANCE_BNF_DESCR", "CHEMICAL_SUBSTANCE_BNF_DESCR_INDEX").show(30, truncate=False)



+-----------------------------------+----------------------------------+
|CHEMICAL_SUBSTANCE_BNF_DESCR       |CHEMICAL_SUBSTANCE_BNF_DESCR_INDEX|
+-----------------------------------+----------------------------------+
|Metformin hydrochloride            |0.0                               |
|Metformin hydrochloride            |0.0                               |
|Metformin hydrochloride            |0.0                               |
|Metformin hydrochloride            |0.0                               |
|Metformin hydrochloride            |0.0                               |
|Metformin hydrochloride            |0.0                               |
|Metformin hydrochloride            |0.0                               |
|Metformin hydrochloride/sitagliptin|1.0                               |
|Metformin hydrochloride            |0.0                               |
|Metformin hydrochloride            |0.0                               |
|Metformin hydrochloride            |0.0           

In [22]:
# Drop the specified columns
columns_to_drop = ["BNF_DESCRIPTION", "BNF_DESCRIPTION_INDEX", "CHEMICAL_SUBSTANCE_BNF_DESCR"]
df_encoded = df_encoded.drop(*columns_to_drop)

# Verify the updated DataFrame
df_encoded.printSchema()  # Check remaining columns
df_encoded.show(truncate=False)  # Display a few rows of the updated DataFrame


root
 |-- TOTAL_QUANTITY: integer (nullable = true)
 |-- ITEMS: integer (nullable = true)
 |-- NIC: double (nullable = true)
 |-- ADQUSAGE: double (nullable = true)
 |-- ACTUAL_COST: double (nullable = true)
 |-- QUANTITY: integer (nullable = true)
 |-- PRACTICE_LATITUDE: double (nullable = false)
 |-- PRACTICE_LONGITUDE: double (nullable = false)
 |-- YEAR: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- BNF_DESCRIPTION_OHE: vector (nullable = true)
 |-- CHEMICAL_SUBSTANCE_BNF_DESCR_INDEX: double (nullable = false)

+--------------+-----+------+----------+-----------+--------+-----------------+------------------+----+-----+-------------------+----------------------------------+
|TOTAL_QUANTITY|ITEMS|NIC   |ADQUSAGE  |ACTUAL_COST|QUANTITY|PRACTICE_LATITUDE|PRACTICE_LONGITUDE|YEAR|MONTH|BNF_DESCRIPTION_OHE|CHEMICAL_SUBSTANCE_BNF_DESCR_INDEX|
+--------------+-----+------+----------+-----------+--------+-----------------+------------------+----+-----+-----------------

In [28]:
from pyspark.sql.functions import log, col

# Apply log transformation to the target variable
df_encoded = df_encoded.withColumn("LOG_ACTUAL_COST", log(col("ACTUAL_COST") + 1))

# Display transformed target variable
df_encoded.select("ACTUAL_COST", "LOG_ACTUAL_COST").show(truncate=False)


+-----------+------------------+
|ACTUAL_COST|LOG_ACTUAL_COST   |
+-----------+------------------+
|415.30817  |6.031425779278594 |
|3.70971    |1.5496263350074813|
|136.4584   |4.923321325875971 |
|37.02111   |3.6381415317945547|
|1.56749    |0.9429287679736535|
|1.69559    |0.9916171043338229|
|6.67305    |2.0377141895303503|
|7.98134    |2.195149091671816 |
|3.12257    |1.41647675529533  |
|15.96268   |2.831015636748015 |
|10.19225   |2.41522157449755  |
|4.15933    |1.6408067261016441|
|11.46614   |2.5230161688706847|
|40.07547   |3.7154111063411293|
|3.70954    |1.54959023871531  |
|11.2415    |2.50483181859527  |
|22.33937   |3.150141634206194 |
|57.88679   |4.07561678706534  |
|24.931     |3.2554391641752476|
|6.98219    |2.0772128099012477|
+-----------+------------------+
only showing top 20 rows



In [30]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.sql.functions import log, col

# Step 1: Drop existing conflicting columns
columns_to_drop = [
    "TOTAL_QUANTITY_vec", "TOTAL_QUANTITY_scaled",
    "NIC_vec", "NIC_scaled",
    "ADQUSAGE_vec", "ADQUSAGE_scaled"
]
df_encoded = df_encoded.drop(*columns_to_drop)

# Step 2: Log-transform the selected columns
log_columns = ["TOTAL_QUANTITY", "NIC", "ADQUSAGE"]
for col_name in log_columns:
    df_encoded = df_encoded.withColumn(f"LOG_{col_name}", log(col(col_name) + 1))

# Step 3: Apply separate scaling for each log-transformed column
for col_name in log_columns:
    # Assemble the column into a vector
    assembler = VectorAssembler(inputCols=[f"LOG_{col_name}"], outputCol=f"{col_name}_vec")
    df_encoded = assembler.transform(df_encoded)

    # Apply standard scaling
    scaler = StandardScaler(inputCol=f"{col_name}_vec", outputCol=f"{col_name}_scaled", withMean=True, withStd=True)
    scaler_model = scaler.fit(df_encoded)
    df_encoded = scaler_model.transform(df_encoded)

# Step 4: Display scaled columns
df_encoded.select(
    "LOG_TOTAL_QUANTITY", "TOTAL_QUANTITY_scaled",
    "LOG_NIC", "NIC_scaled",
    "LOG_ADQUSAGE", "ADQUSAGE_scaled"
).show(truncate=False)


+------------------+----------------------+------------------+----------------------+------------------+----------------------+
|LOG_TOTAL_QUANTITY|TOTAL_QUANTITY_scaled |LOG_NIC           |NIC_scaled            |LOG_ADQUSAGE      |ADQUSAGE_scaled       |
+------------------+----------------------+------------------+----------------------+------------------+----------------------+
|9.660205067381032 |[2.541626137164313]   |6.094472297182211 |[2.455755774083107]   |8.56172031219105  |[2.461121069971588]   |
|4.727387818712341 |[-0.8226463058090829] |1.5769147207285403|[-0.9202412683543075] |3.6463197527386146|[-0.9033930524759506] |
|8.547334348328224 |[1.7826277587606107]  |4.986069340151007 |[1.6274397999226375]  |7.449110105843086 |[1.6995569396916341]  |
|6.970730078143525 |[0.707354536123367]   |3.4468078929142076|[0.47714077097507374] |5.873993972005715 |[0.6214147999631158]  |
|4.04305126783455  |[-1.2893764690761236] |0.9783261227936078|[-1.3675700078479758] |2.9789253247291207|

In [42]:
# List of columns to drop
columns_to_drop = [
    "ITEMS_vec"
]
# Drop the unnecessary columns
df_encoded = df_encoded.drop(*columns_to_drop)

In [44]:
from pyspark.ml.feature import MinMaxScaler, VectorAssembler

# Step 1: Assemble the ITEMS column into a vector
assembler = VectorAssembler(inputCols=["ITEMS"], outputCol="ITEMS_vec")
df_encoded = assembler.transform(df_encoded)

# Step 2: Apply MinMaxScaler
scaler = MinMaxScaler(inputCol="ITEMS_vec", outputCol="ITEMS_scaled")
scaler_model = scaler.fit(df_encoded)
df_encoded = scaler_model.transform(df_encoded)

# Step 3: Display the original and scaled values
df_encoded.select("ITEMS", "ITEMS_scaled").show(truncate=False)



+-----+-----------------------+
|ITEMS|ITEMS_scaled           |
+-----+-----------------------+
|70   |[0.09324324324324325]  |
|1    |[0.0]                  |
|23   |[0.02972972972972973]  |
|76   |[0.10135135135135136]  |
|1    |[0.0]                  |
|1    |[0.0]                  |
|1    |[0.0]                  |
|12   |[0.014864864864864866] |
|1    |[0.0]                  |
|24   |[0.031081081081081083] |
|8    |[0.00945945945945946]  |
|4    |[0.004054054054054054] |
|18   |[0.022972972972972974] |
|9    |[0.010810810810810811] |
|12   |[0.014864864864864866] |
|6    |[0.006756756756756757] |
|2    |[0.0013513513513513514]|
|13   |[0.016216216216216217] |
|4    |[0.004054054054054054] |
|1    |[0.0]                  |
+-----+-----------------------+
only showing top 20 rows



In [46]:
from pyspark.ml.feature import MinMaxScaler

# List of individual columns to scale
columns_to_scale = ["QUANTITY", "YEAR", "MONTH"]

# Scale each column separately
for column in columns_to_scale:
    # Assemble the column into a vector
    assembler = VectorAssembler(inputCols=[column], outputCol=f"{column}_vec")
    df_encoded = assembler.transform(df_encoded)

    # Apply Min-Max Scaling
    scaler = MinMaxScaler(inputCol=f"{column}_vec", outputCol=f"{column}_scaled")
    scaler_model = scaler.fit(df_encoded)
    df_encoded = scaler_model.transform(df_encoded)

# Display scaled columns
df_encoded.select("QUANTITY", "QUANTITY_scaled", "YEAR", "YEAR_scaled", "MONTH", "MONTH_scaled").show(truncate=False)


+--------+-----------------------+----+-----------+-----+--------------------+
|QUANTITY|QUANTITY_scaled        |YEAR|YEAR_scaled|MONTH|MONTH_scaled        |
+--------+-----------------------+----+-----------+-----+--------------------+
|224     |[0.1859883236030025]   |2023|[0.0]      |7    |[0.5454545454545454]|
|112     |[0.09257714762301918]  |2023|[0.0]      |7    |[0.5454545454545454]|
|224     |[0.1859883236030025]   |2023|[0.0]      |7    |[0.5454545454545454]|
|14      |[0.010842368640533777] |2023|[0.0]      |7    |[0.5454545454545454]|
|56      |[0.04587155963302752]  |2023|[0.0]      |7    |[0.5454545454545454]|
|60      |[0.04920767306088407]  |2023|[0.0]      |7    |[0.5454545454545454]|
|252     |[0.2093411175979983]   |2023|[0.0]      |7    |[0.5454545454545454]|
|21      |[0.016680567139282735] |2023|[0.0]      |7    |[0.5454545454545454]|
|112     |[0.09257714762301918]  |2023|[0.0]      |7    |[0.5454545454545454]|
|21      |[0.016680567139282735] |2023|[0.0]      |7

In [48]:
from pyspark.ml.clustering import KMeans

# Assemble latitude and longitude for clustering
geo_assembler = VectorAssembler(inputCols=["PRACTICE_LATITUDE", "PRACTICE_LONGITUDE"], outputCol="geo_features")
df_encoded = geo_assembler.transform(df_encoded)

# Apply KMeans clustering with k=4
kmeans = KMeans(featuresCol="geo_features", predictionCol="geo_cluster", k=4, seed=42)
kmeans_model = kmeans.fit(df_encoded)
df_encoded = kmeans_model.transform(df_encoded)

# Display geolocation clusters
df_encoded.select("PRACTICE_LATITUDE", "PRACTICE_LONGITUDE", "geo_cluster").show()


+-----------------+------------------+-----------+
|PRACTICE_LATITUDE|PRACTICE_LONGITUDE|geo_cluster|
+-----------------+------------------+-----------+
|       51.5135999|          -0.05055|          1|
|      51.51835505|       -0.03830635|          2|
|       51.5363018|      -0.027021075|          2|
|       51.5238708|         -0.014907|          3|
|         51.52979|          -0.03887|          2|
|       51.5141762|        -0.0143071|          0|
|       51.5141762|        -0.0143071|          0|
|      51.50533401|      -0.058222367|          1|
|       51.5139726|        -0.0550705|          1|
|      51.52468015|       -0.04266905|          2|
|       51.5173976|        -0.0696235|          1|
|         51.52377|        -0.0651999|          1|
|       51.5269199|          -0.06426|          1|
|       51.5269199|          -0.06426|          1|
|       51.5238708|         -0.014907|          3|
|      51.52589465|       -0.02501745|          3|
|      51.51448982|      -0.004

In [58]:
# List of columns to drop
columns_to_drop = [
    "TOTAL_QUANTITY", "NIC", "ADQUSAGE",  # Original columns (log-transformed versions exist)
    "LOG_TOTAL_QUANTITY", "LOG_NIC", "LOG_ADQUSAGE",  # Intermediate log-transformed columns
    "TOTAL_QUANTITY_vec", "NIC_vec", "ADQUSAGE_vec", # Vectorized columns no longer needed
    "QUANTITY_vec", "YEAR_vec", "Month_vec", "geo_features",
    "ITEMS_vec","ITEMS", "ITEMS_minmax_scaled"
]
# Drop the unnecessary columns
df_encoded = df_encoded.drop(*columns_to_drop)

# Show the resulting DataFrame's schema
df_encoded.printSchema()


root
 |-- ACTUAL_COST: double (nullable = true)
 |-- BNF_DESCRIPTION_OHE: vector (nullable = true)
 |-- CHEMICAL_SUBSTANCE_BNF_DESCR_INDEX: double (nullable = false)
 |-- LOG_ACTUAL_COST: double (nullable = true)
 |-- TOTAL_QUANTITY_scaled: vector (nullable = true)
 |-- NIC_scaled: vector (nullable = true)
 |-- ADQUSAGE_scaled: vector (nullable = true)
 |-- ITEMS_scaled: vector (nullable = true)
 |-- QUANTITY_scaled: vector (nullable = true)
 |-- YEAR_scaled: vector (nullable = true)
 |-- MONTH_scaled: vector (nullable = true)
 |-- geo_cluster: integer (nullable = false)



In [60]:
df_encoded.show(truncate=False)

+-----------+-------------------+----------------------------------+------------------+----------------------+----------------------+----------------------+-----------------------+-----------------------+-----------+--------------------+-----------+
|ACTUAL_COST|BNF_DESCRIPTION_OHE|CHEMICAL_SUBSTANCE_BNF_DESCR_INDEX|LOG_ACTUAL_COST   |TOTAL_QUANTITY_scaled |NIC_scaled            |ADQUSAGE_scaled       |ITEMS_scaled           |QUANTITY_scaled        |YEAR_scaled|MONTH_scaled        |geo_cluster|
+-----------+-------------------+----------------------------------+------------------+----------------------+----------------------+----------------------+-----------------------+-----------------------+-----------+--------------------+-----------+
|415.30817  |(27,[0],[1.0])     |0.0                               |6.031425779278594 |[2.541626137164313]   |[2.455755774083107]   |[2.461121069971588]   |[0.09324324324324325]  |[0.1859883236030025]   |[0.0]      |[0.5454545454545454]|1          |


In [62]:
from pyspark.sql.functions import col, count, when, isnull

# Create a list of columns with their null counts and percentages
total_rows = df_encoded.count()

# Count nulls for each column
null_counts = df_encoded.select([
    count(when(isnull(c) | col(c).isNull(), c)).alias(c)
    for c in df_encoded.columns
])

# Display null counts and percentages for each column
for column in null_counts.first().asDict().items():
    column_name = column[0]
    null_count = column[1]
    null_percentage = (null_count/total_rows) * 100
    print(f"Column: {column_name}")
    print(f"Null Count: {null_count}")
    print(f"Null Percentage: {null_percentage:.2f}%")
    print("-" * 50)

Column: ACTUAL_COST
Null Count: 0
Null Percentage: 0.00%
--------------------------------------------------
Column: BNF_DESCRIPTION_OHE
Null Count: 0
Null Percentage: 0.00%
--------------------------------------------------
Column: CHEMICAL_SUBSTANCE_BNF_DESCR_INDEX
Null Count: 0
Null Percentage: 0.00%
--------------------------------------------------
Column: LOG_ACTUAL_COST
Null Count: 0
Null Percentage: 0.00%
--------------------------------------------------
Column: TOTAL_QUANTITY_scaled
Null Count: 0
Null Percentage: 0.00%
--------------------------------------------------
Column: NIC_scaled
Null Count: 0
Null Percentage: 0.00%
--------------------------------------------------
Column: ADQUSAGE_scaled
Null Count: 0
Null Percentage: 0.00%
--------------------------------------------------
Column: ITEMS_scaled
Null Count: 0
Null Percentage: 0.00%
--------------------------------------------------
Column: QUANTITY_scaled
Null Count: 0
Null Percentage: 0.00%
--------------------------

In [49]:
spark.stop

<bound method SparkSession.stop of <pyspark.sql.session.SparkSession object at 0x00000227168407F0>>